# 01. Сборка панели «МО × месяц»

Ноутбук собирает все источники в один датафрейм с ключом **(`mo_id`, `date`)** и строит признаки для прогноза.

**Соглашение о времени (важно для отсутствия утечки).** Строка с датой `t` содержит только то, что известно
**на конец месяца `t`**: текущие расходы `y`, их лаги и внешние признаки с учётом лага публикации
(например, ИПЦ за месяц `t` Росстат публикует в середине `t+1`, поэтому в строку `t` попадает ИПЦ за `t-1`).
Цели для горизонтов прогноза лежат в колонках `target_h1`, `target_h3`, `target_h6`, `target_h12` — это
расходы через `h` месяцев. Модель учится по признакам строки `t` предсказывать `target_h{h}`.

**Ключ МО.** В открытом файле СберИндекса нет id, только названия. `mo_id` — это код ОКТМО, найденный
сопоставлением названия со справочником Росстата (бюллетень «Численность населения по МО на 1 января 2024»).
Названия, которые нельзя однозначно сопоставить (одно имя у нескольких МО), исключаются и пишутся в лог.

Структура: `data/raw/` — исходники, `data/processed/` — результат. Запуск: Run All.

In [ ]:
import re, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import holidays

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW, OUT = ROOT / "data" / "raw", ROOT / "data" / "processed"
OUT.mkdir(parents=True, exist_ok=True)

FILES = {
    "sber_mo":      "potrebitelskie-beznalicnye-rashody-na-urovne-munizipalnyh-obrazovanij_ru_1764079373653.parquet",
    "sber_spend":   "consumer-spending_ru_1791179920805.parquet",
    "sber_growth":  "consumer-spending-growth_ru_1791179934458.parquet",
    "sber_sa":      "consumper-spending-index-sa_ru_1791179947372.parquet",
    "sber_age_w":   "potrebitelskaya-aktivnost-po-kategoriyam-tovarov-v-razreze-vozrastov_ru_1790951484969.parquet",
    "sber_cat_w":   "ver-izmenenie-trat-po-kategoriyam_ru_1791191909581.parquet",
    "bul_2023":     "BUL_MO_2023.xlsx",
    "bul_2024":     "BUL_MO_2024.xlsx",
    "cpi_rf":       "ipc_mes_08-2026.xlsx",
    "cpi_reg":      "ipc_RF_fo_sub_08-2026.xlsx",
    "wage_reg":     "tab2-zpl_07-2026.xlsx",
    "unemp_reg":    "trud_3_15-72.xlsx",
    "loans_reg":    "02_04_New_loans_ind.xlsx",
    "key_rate":     "key_rate_monthly_2023_2026.parquet",
    "infl_exp":     "cbr_inflation_expectations_key.parquet",
}

# Лаг публикации (мес.): сколько месяцев проходит, пока показатель за месяц m становится известен.
PUB_LAG = {"cpi": 1, "wage": 2, "unemp": 1, "loans": 1, "key_rate": 0, "infl_exp": 0,
           "sber_nat_month": 1, "sber_nat_week": 0}

TARGET_CAT = "Все категории"
HORIZONS = [1, 3, 6, 12]
Y_LAGS = [1, 2, 3, 6, 12]
START, END = "2022-01-01", "2024-12-01"   # 2022 нужен только как история для лагов внешних признаков

MONTHS = {m: i for i, m in enumerate(["январь", "февраль", "март", "апрель", "май", "июнь", "июль",
          "август", "сентябрь", "октябрь", "ноябрь", "декабрь"], 1)}
def f(name): return RAW / FILES[name]

## 1. Нормализация названий регионов
Источники по-разному пишут регионы («Еврейская авт.область», «Республика Татарстан (Татарстан)»,
«Кемеровская область - Кузбасс»). Приводим к ключу: нижний регистр, без скобок и суффиксов, токены по алфавиту.

In [ ]:
def region_key(s):
    s = str(s).lower().replace("ё", "е").replace("\n", " ")
    s = re.sub(r"\(.*?\)", " ", s)
    s = re.sub(r",?\s*включая.*$", " ", s)                       # «Архангельская область, включая Ненецкий АО»
    s = re.sub(r"\s[-–—]\s.*$", " ", s)                       # «… - Кузбасс», «… - город федерального значения»
    s = re.sub(r"-(алания|кузбасс|югра|чувашия)\b", " ", s)
    s = s.replace("авт.", "авт ").replace("автономная", "авт").replace("автономный", "авт")
    s = re.sub(r"\b(г|город|федерального|значения)\b", " ", s)
    s = re.sub(r"[^\w\s-]", " ", s)
    return " ".join(sorted(s.split()))

SKIP_REGION = re.compile(r"федеральный округ|российская федерация|^рф$", re.I)

def pick_regions(df, name_col="name"):
    """Оставляет строки субъектов; при дублях ключа (Тюменская/Архангельская с АО и без) берёт вариант «кроме/без»."""
    df = df[~df[name_col].astype(str).str.contains(SKIP_REGION)].copy()
    df["region_key"] = df[name_col].map(region_key)
    df["_pref"] = df[name_col].astype(str).str.contains("кроме|без", case=False).astype(int)
    return df.sort_values("_pref").drop_duplicates("region_key", keep="last").drop(columns="_pref")

## 2. Справочник МО (ОКТМО, регион, население)

In [ ]:
def load_bulletin_2024():
    subj = pd.read_excel(f("bul_2024"), "Перечень_субъектов_РФ", header=None).iloc[1:, 1].dropna().astype(str).str.strip().tolist()
    subj = sorted(subj, key=len, reverse=True)
    d = pd.read_excel(f("bul_2024"), "Численность_по_МО", header=None, skiprows=7).iloc[:, :5]
    d.columns = ["code", "name", "pop", "pop_urban", "pop_rural"]
    d = d[d.name.notna() & pd.to_numeric(d["pop"], errors="coerce").notna()].copy()
    d["name"] = d.name.astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
    d["code"] = d.code.astype(str).str.replace(r"\.0$", "", regex=True).str.replace(" ", "")
    cur, regs, hdr = None, [], []
    for n in d.name:                                          # регион = последний встреченный заголовок субъекта
        r = next((s for s in subj if n.startswith(s.split(" - ")[0] if s.startswith("г ") else s)), None)
        hdr.append(r is not None); cur = r or cur; regs.append(cur)
    d["region"], d["_hdr"] = regs, hdr
    d = d[~d._hdr & d.code.str.fullmatch(r"\d{10,11}")].drop(columns="_hdr")
    d["is_top"] = d.code.str[5:8].eq("000")                   # МО верхнего уровня (район / округ / ГО)
    return d

def load_bulletin_2023(regions):
    e = pd.read_excel(f("bul_2023"), "Численность_по_МО", header=None, skiprows=8).iloc[:, :2]
    e.columns = ["name", "pop_2023"]
    e = e[e.name.notna() & pd.to_numeric(e.pop_2023, errors="coerce").notna()].copy()
    e["name"] = e.name.astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
    cur, regs = None, []
    for n in e.name:
        if n in regions: cur = n
        regs.append(cur)
    e["region"] = regs
    return e[~e.name.isin(regions)].drop_duplicates(["region", "name"])

ref = load_bulletin_2024().rename(columns={"pop": "pop_2024"})
ref = ref.merge(load_bulletin_2023(set(ref.region)), on=["region", "name"], how="left")
ref["region_key"] = ref.region.map(region_key)
top = ref[ref.is_top].copy()
print(f"МО верхнего уровня: {len(top)}, регионов: {top.region.nunique()}, население 2023 найдено у {top.pop_2023.notna().mean():.0%}")

## 3. Сопоставление названий МО СберИндекса с ОКТМО

In [ ]:
STOP = r"\b(внутригородская|территория|города|федерального|значения|городской|муниципальный|муниипальный|" \
       r"муниципального|национальный|район|округ|поселение|образования|город|герой|им|имени|г|рп|пгт|п|поселок|" \
       r"рабочий|сельское|городское|зато|закрытое|административно|территориальное|типа)\b"
def mo_key(s):
    s = str(s).lower().replace("ё", "е")
    s = re.sub(r"[«»\"().,№]", " ", s).replace("-", " ")
    return re.sub(r"\s+", " ", re.sub(STOP, " ", s)).strip()
def mo_type(s):
    s = str(s).lower()
    if "внутригородская" in s: return "city_part"
    if "муниципальный район" in s: return "mr"
    if "муниципальный округ" in s or "муниипальный округ" in s: return "mo"
    if "городской округ" in s or s.startswith("г "): return "go"
    return "other"

sber = pd.read_parquet(f("sber_mo"))
sber["date"] = pd.to_datetime(sber.period)
names = pd.DataFrame({"mo_name": sber.mo.unique()})
names["k"], names["t"] = names.mo_name.map(mo_key), names.mo_name.map(mo_type)
# у Сбера под одним названием бывает несколько разных МО (несколько значений на одну дату) — такие не трогаем
n_dup = sber.groupby(["mo", "category_15", "date"]).size().groupby("mo").max()
names["sber_dup"] = names.mo_name.map(n_dup) > 1

top["k"], top["t"] = top.name.map(mo_key), top.name.map(mo_type)
fed = top.region.str.contains("федерального значения")
top.loc[fed, "t"] = "city_part"

def unique_match(left, right, on):
    m = left.merge(right[on + ["code"]], on=on, how="inner")
    cnt = m.groupby("mo_name").code.nunique()
    return m[m.mo_name.isin(cnt[cnt == 1].index)].drop_duplicates("mo_name")[["mo_name", "code"]]

todo = names[~names.sber_dup]
m1 = unique_match(todo, top, ["k", "t"])                                   # шаг 1: имя + тип МО
m2 = unique_match(todo[~todo.mo_name.isin(m1.mo_name)], top, ["k"])        # шаг 2: только имя
match = pd.concat([m1, m2]).rename(columns={"code": "mo_id"})
match = match[~match.mo_id.duplicated(keep=False)]                          # один ОКТМО — одно название

names = names.merge(match, on="mo_name", how="left")
names["status"] = np.select([names.sber_dup, names.mo_id.notna()], ["дубль названия у Сбера", "ok"], "не найдено/неоднозначно")
print(names.status.value_counts().to_string())
names.to_csv(OUT / "mo_matching_log.csv", index=False)

## 4. Таргет: расходы по МО (широкий формат по категориям)

In [ ]:
sb = sber.merge(match, left_on="mo", right_on="mo_name", how="inner")
cat_map = {"Все категории": "all", "Продовольствие": "food", "Общественное питание": "catering",
           "Здоровье": "health", "Маркетплейсы": "marketplaces", "Транспорт": "transport"}
y = sb.pivot_table(index=["mo_id", "date"], columns="category_15", values="value", aggfunc="first")
y.columns = ["y_" + cat_map[c] for c in y.columns]
y = y.reset_index()

# полная сетка МО × месяц (пропуски остаются NaN — модель/скрипты оценки их отфильтруют)
grid = pd.MultiIndex.from_product([y.mo_id.unique(), pd.date_range(START, END, freq="MS")], names=["mo_id", "date"])
panel = pd.DataFrame(index=grid).reset_index().merge(y, on=["mo_id", "date"], how="left")
panel = panel.merge(match.rename(columns={"mo_name": "mo_name_sber"}), on="mo_id", how="left")
print(panel.shape, "| МО:", panel.mo_id.nunique(), "| месяцев с таргетом:", panel.dropna(subset=["y_all"]).date.nunique())

## 5. Статические признаки МО

In [ ]:
stat = top[["code", "name", "region", "region_key", "pop_2024", "pop_2023", "pop_urban", "t"]].rename(
    columns={"code": "mo_id", "name": "mo_name_rosstat", "t": "mo_type"})
panel = panel.merge(stat, on="mo_id", how="left")
# население на 1 января года наблюдения (2022 → берём 2023 как ближайшее)
panel["population"] = np.where(panel.date.dt.year >= 2024, panel.pop_2024, panel.pop_2023.fillna(panel.pop_2024))
panel["log_population"] = np.log1p(panel.population)
panel["urban_share"] = panel.pop_urban / panel.pop_2024
panel["pop_change_pct"] = (panel.pop_2024 / panel.pop_2023 - 1) * 100          # отток / приток за 2023 год
panel["size_class"] = pd.cut(panel.pop_2024, [0, 10_000, 50_000, 250_000, 1_000_000, np.inf],
                             labels=["<10k", "10-50k", "50-250k", "250k-1M", ">1M"]).astype(str)
panel = panel.drop(columns=["pop_2024", "pop_2023", "pop_urban"])

## 6. Региональные признаки Росстата и ЦБ

In [ ]:
def load_cpi_regional():
    xl = pd.ExcelFile(f("cpi_reg")); rows = []
    for sh in xl.sheet_names:
        mm = re.fullmatch(r"(\d{2})\((\d{4})\)", sh)
        if not mm: continue
        d = pd.read_excel(xl, sh, header=None, skiprows=4).iloc[:, :6]
        d.columns = ["code", "name", "cpi", "cpi_food", "cpi_nonfood", "cpi_services"]
        d = d[pd.to_numeric(d.cpi, errors="coerce").notna()]
        d["date"] = pd.Timestamp(int(mm.group(2)), int(mm.group(1)), 1); rows.append(d)
    d = pd.concat(rows)
    out = []
    for dt, g in d.groupby("date"): out.append(pick_regions(g))
    d = pd.concat(out)
    for c in ["cpi", "cpi_food", "cpi_nonfood", "cpi_services"]: d[c] = pd.to_numeric(d[c], errors="coerce") - 100  # % м/м
    return d[["region_key", "date", "cpi", "cpi_food", "cpi_nonfood", "cpi_services"]]

def load_wages():
    d = pd.read_excel(f("wage_reg"), "с 2019", header=None)
    years = d.iloc[1, 1:].ffill().astype(str).str.extract(r"(\d{4})")[0]
    months = d.iloc[2, 1:].astype(str).str.lower().str.replace(r"[^а-я]", "", regex=True).map(MONTHS)  # «октябрь1)» → октябрь
    dates = [pd.Timestamp(int(yv), int(mv), 1) if pd.notna(yv) and pd.notna(mv) else pd.NaT for yv, mv in zip(years, months)]
    body = d.iloc[3:].rename(columns={0: "name"})
    body = pick_regions(body[body.name.notna()])
    long = body.melt(id_vars=["name", "region_key"], var_name="col", value_name="wage")
    long["date"] = long.col.map(dict(zip(d.columns[1:], dates)))
    long["wage"] = pd.to_numeric(long.wage, errors="coerce")
    return long.dropna(subset=["date", "wage"])[["region_key", "date", "wage"]]

def load_unemployment():
    d = pd.read_excel(f("unemp_reg"), "4", header=None)
    hdr = d.iloc[4, 1:].astype(str).str.replace("\n", " ").str.lower()
    def end_month(s):  # «ноябрь 2016 - январь 20171)» → январь 2017
        mm = re.findall(r"([а-я]+)\s*(\d{4})", s)
        return pd.Timestamp(int(mm[-1][1]), MONTHS[mm[-1][0]], 1) if mm and mm[-1][0] in MONTHS else pd.NaT
    dates = dict(zip(d.columns[1:], hdr.map(end_month)))
    body = d.iloc[5:].rename(columns={0: "name"}); body = pick_regions(body[body.name.notna()])
    long = body.melt(id_vars=["name", "region_key"], var_name="col", value_name="unemp")
    long["date"] = long.col.map(dates); long["unemp"] = pd.to_numeric(long.unemp, errors="coerce")
    return long.dropna(subset=["date", "unemp"])[["region_key", "date", "unemp"]]

RU_MONTHS_CAP = {k.capitalize(): v for k, v in MONTHS.items()}
def load_loans():
    d = pd.read_excel(f("loans_reg"), "итого", header=None)
    def parse(s):
        mm = re.match(r"([А-Яа-я]+)\s+(\d{4})", str(s).strip())
        return pd.Timestamp(int(mm.group(2)), RU_MONTHS_CAP[mm.group(1).capitalize()], 1) if mm else pd.NaT
    dates = dict(zip(d.columns[1:], d.iloc[2, 1:].map(parse)))
    body = d.iloc[3:].rename(columns={0: "name"}); body = body[body.name.notna()]
    body = pick_regions(body[~body.name.astype(str).str.isupper()])       # ФО и РФ в файле ЦБ капсом
    long = body.melt(id_vars=["name", "region_key"], var_name="col", value_name="loans")
    long["date"] = long.col.map(dates); long["loans"] = pd.to_numeric(long.loans, errors="coerce")
    return long.dropna(subset=["date", "loans"])[["region_key", "date", "loans"]]

reg = (load_cpi_regional()
       .merge(load_wages(), on=["region_key", "date"], how="outer")
       .merge(load_unemployment(), on=["region_key", "date"], how="outer")
       .merge(load_loans(), on=["region_key", "date"], how="outer")
       .sort_values(["region_key", "date"]))
unmatched = sorted(set(panel.region_key.dropna()) - set(reg.region_key))
print("регионов МО без региональных данных:", unmatched)

# производные: накопленный ИПЦ, г/г, реальная зарплата
g = reg.groupby("region_key")
reg["cpi_index"] = g.cpi.transform(lambda s: (1 + s.fillna(0) / 100).cumprod())
reg["cpi_yoy"] = (reg.cpi_index / g.cpi_index.shift(12) - 1) * 100
reg["wage_yoy"] = (reg.wage / g.wage.shift(12) - 1) * 100
reg["real_wage_yoy"] = ((1 + reg.wage_yoy / 100) / (1 + reg.cpi_yoy / 100) - 1) * 100
reg["loans_yoy"] = (reg.loans / g.loans.shift(12) - 1) * 100
reg = reg.drop(columns="cpi_index")

# сдвиг на лаг публикации: в строку t попадает то, что известно на конец t
REG_LAG = {"cpi": "cpi", "cpi_food": "cpi", "cpi_nonfood": "cpi", "cpi_services": "cpi", "cpi_yoy": "cpi",
           "wage": "wage", "wage_yoy": "wage", "real_wage_yoy": "wage", "unemp": "unemp",
           "loans": "loans", "loans_yoy": "loans"}
reg_feats = reg[["region_key", "date"]].copy()
for col, src in REG_LAG.items():
    reg_feats[f"reg_{col}_L{PUB_LAG[src]}"] = reg.groupby("region_key")[col].shift(PUB_LAG[src])
    reg_feats[f"reg_{col}_L{PUB_LAG[src] + 2}"] = reg.groupby("region_key")[col].shift(PUB_LAG[src] + 2)
panel = panel.merge(reg_feats, on=["region_key", "date"], how="left")

## 7. Национальные признаки: ИПЦ России, ключевая ставка, инфляционные ожидания, СберИндекс

In [ ]:
def load_cpi_rf():
    out = {}
    for sh, name in [("01", "cpi"), ("02", "cpi_food"), ("03", "cpi_nonfood"), ("04", "cpi_services")]:
        d = pd.read_excel(f("cpi_rf"), sh, header=None)
        yrow = d.index[d.iloc[:, 1:].apply(pd.to_numeric, errors="coerce").ge(1990).sum(axis=1) > 5][0]
        years = pd.to_numeric(d.iloc[yrow, 1:], errors="coerce")
        start = d.index[d[0].astype(str).str.contains("предыдущего месяца")][0] + 1
        block = d.iloc[start:start + 12]
        for (_, r) in block.iterrows():
            mth = MONTHS.get(str(r[0]).strip().lower())
            if not mth: continue
            for c, yv in years.dropna().items():
                v = pd.to_numeric(r[c], errors="coerce")
                if pd.notna(v): out.setdefault(name, {})[pd.Timestamp(int(yv), mth, 1)] = v - 100
    return pd.DataFrame(out).sort_index().rename_axis("date").reset_index()

nat = load_cpi_rf()
kr = pd.read_parquet(f("key_rate")).rename(columns={"month": "date", "key_rate_pct": "key_rate"})
kr["date"] = pd.to_datetime(kr.date)
ie = pd.read_parquet(f("infl_exp")).reset_index().rename(columns={"period": "date"})
ie = ie[["date", "exp_1y_median", "obs_median", "exp_1y_savers", "exp_1y_nosavers", "ipn", "idx_expectations", "idx_current"]]

def sber_nat_month():
    sp = pd.read_parquet(f("sber_spend")); sp["date"] = pd.to_datetime(sp.period)
    a = sp.pivot_table(index="date", columns="type", values="value").add_prefix("sber_nat_spend_")
    gr = pd.read_parquet(f("sber_growth")); gr["date"] = pd.to_datetime(gr.period)
    b = gr.pivot_table(index="date", columns=["value_type", "type"], values="value")
    b.columns = [f"sber_nat_yoy_{v}_{t}" for v, t in b.columns]
    sa = pd.read_parquet(f("sber_sa")); sa["date"] = pd.to_datetime(sa.period)
    c = sa.pivot_table(index="date", columns="type", values="value").add_prefix("sber_nat_sa_")
    return pd.concat([a, b, c], axis=1)

def sber_nat_week():
    """Недельные ряды → среднее за месяц: публикуются почти в реальном времени, это ранний сигнал."""
    ag = pd.read_parquet(f("sber_age_w")); ag["date"] = pd.to_datetime(ag.period).dt.to_period("M").dt.to_timestamp()
    a = ag.pivot_table(index="date", columns="age", values="value", aggfunc="mean").add_prefix("sber_w_age_")
    ct = pd.read_parquet(f("sber_cat_w")); ct["date"] = pd.to_datetime(ct.period).dt.to_period("M").dt.to_timestamp()
    b = ct.pivot_table(index="date", columns="category", values="value", aggfunc="mean").add_prefix("sber_w_cat_yoy_")
    return pd.concat([a, b], axis=1)

def clean_cols(df):
    df.columns = [re.sub(r"[^\w]+", "_", c.lower()).strip("_") for c in df.columns]; return df

nat_parts = {
    "cpi": nat.set_index("date"), "key_rate": kr.set_index("date"), "infl_exp": ie.set_index("date"),
    "sber_nat_month": sber_nat_month(), "sber_nat_week": sber_nat_week(),
}
nat_feats = []
for src, df in nat_parts.items():
    df = df.sort_index().shift(PUB_LAG[src], freq="MS")       # лаг публикации
    df = clean_cols(df.add_prefix("nat_" if not src.startswith("sber") else "").add_suffix(f"_L{PUB_LAG[src]}"))
    nat_feats.append(df)
nat_feats = pd.concat(nat_feats, axis=1).rename_axis("date").reset_index()
panel = panel.merge(nat_feats, on="date", how="left")
print("национальных признаков:", nat_feats.shape[1] - 1)

## 8. Календарь

In [ ]:
ru_hol = holidays.Russia(years=range(2021, 2027))
def workdays(d):
    days = pd.date_range(d, d + pd.offsets.MonthEnd(0))
    return int(sum((x.weekday() < 5) and (x not in ru_hol) for x in days))
cal = pd.DataFrame({"date": pd.date_range("2021-01-01", "2026-12-01", freq="MS")})
cal["month"], cal["quarter"], cal["year"] = cal.date.dt.month, cal.date.dt.quarter, cal.date.dt.year
cal["days_in_month"] = cal.date.dt.days_in_month
cal["workdays"] = cal.date.map(workdays)   # приближение: без учёта переносов выходных
cal["is_december"], cal["is_january"] = (cal.month == 12).astype(int), (cal.month == 1).astype(int)
cal["month_sin"], cal["month_cos"] = np.sin(2 * np.pi * cal.month / 12), np.cos(2 * np.pi * cal.month / 12)
def holiday_stats(d):
    days = pd.date_range(d, d + pd.offsets.MonthEnd(0))
    off = [(x.weekday() >= 5) or (x in ru_hol) for x in days]
    run = best = 0
    for o in off: run = run + 1 if o else 0; best = max(best, run)
    return sum(x in ru_hol for x in days), best
cal[["holidays_n", "max_days_off_in_row"]] = cal.date.map(holiday_stats).tolist()
cal["has_long_weekend"] = (cal.max_days_off_in_row >= 4).astype(int)
panel = panel.merge(cal, on="date", how="left")

## 9. Лаги и окна таргета, структура трат, цели для горизонтов

In [ ]:
panel = panel.sort_values(["mo_id", "date"]).reset_index(drop=True)
g = panel.groupby("mo_id")

for L in Y_LAGS:
    panel[f"y_all_lag{L}"] = g.y_all.shift(L)
for w in [3, 6]:
    panel[f"y_all_roll{w}_mean"] = g.y_all.transform(lambda s: s.rolling(w, min_periods=w).mean())
    panel[f"y_all_roll{w}_std"] = g.y_all.transform(lambda s: s.rolling(w, min_periods=w).std())
panel["y_all_diff1"] = panel.y_all - panel.y_all_lag1
panel["y_all_pct1"] = panel.y_all / panel.y_all_lag1 - 1
panel["y_all_yoy"] = panel.y_all / panel.y_all_lag12 - 1
panel["y_all_vs_mo_mean"] = panel.y_all / g.y_all.transform(lambda s: s.expanding().mean())   # без заглядывания вперёд

for c in ["food", "catering", "health", "marketplaces", "transport"]:
    panel[f"share_{c}"] = panel[f"y_{c}"] / panel.y_all
    panel[f"y_{c}_lag1"] = g[f"y_{c}"].shift(1)
    panel[f"share_{c}_lag1"] = g[f"share_{c}"].shift(1)

# относительно региона: насколько МО тратит больше/меньше соседей в этом месяце
panel["y_all_vs_region"] = panel.y_all / panel.groupby(["region_key", "date"]).y_all.transform("median")

# что было в прошлом месяце со всем регионом и с похожими МО (тип × размер): медианный прирост м/м
panel["mo_group"] = panel.mo_type.astype(str) + "|" + panel.size_class.astype(str)
for grp, name in [("region_key", "region"), ("mo_group", "group")]:
    med = panel.groupby([grp, "date"]).y_all_pct1.median().rename(f"{name}_pct1_median").reset_index()
    panel = panel.merge(med, on=[grp, "date"], how="left")
    panel[f"{name}_pct1_median_lag1"] = panel.groupby("mo_id")[f"{name}_pct1_median"].shift(1)
    lvl = panel.groupby([grp, "date"]).y_all.median().rename(f"{name}_y_median").reset_index()
    panel = panel.merge(lvl, on=[grp, "date"], how="left")
panel = panel.sort_values(["mo_id", "date"]).reset_index(drop=True); g = panel.groupby("mo_id")

# цели: расходы через h месяцев (и в логарифмах — удобнее для моделей)
for h in HORIZONS:
    panel[f"target_h{h}"] = g.y_all.shift(-h)
    panel[f"target_date_h{h}"] = panel.date + pd.DateOffset(months=h)

panel["has_target_now"] = panel.y_all.notna().astype(int)
print(panel.shape)

## 10. Проверки и сохранение

In [ ]:
assert not panel.duplicated(["mo_id", "date"]).any(), "дубли ключа"
model_rows = panel[panel.date >= "2023-01-01"]
print("строк с таргетом:", int(model_rows.y_all.notna().sum()))
print("целей доступно:", {h: int(model_rows[f"target_h{h}"].notna().sum()) for h in HORIZONS})
na = model_rows.isna().mean().sort_values(ascending=False)
print("доля пропусков (топ-10):\n", na.head(10).round(2).to_string())

panel.to_parquet(OUT / "panel_mo_month.parquet", index=False)
feature_cols = [c for c in panel.columns if not c.startswith(("target_", "mo_name", "region", "has_target"))
                and c not in ("mo_id", "date")]
pd.Series(feature_cols, name="feature").to_csv(OUT / "feature_list.csv", index=False)
print(f"сохранено: {OUT / 'panel_mo_month.parquet'}  | признаков: {len(feature_cols)}")